<a href="https://colab.research.google.com/github/elvissoares/cbtermo2026-abinitio/blob/main/notebooks/Notebook_Aula01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Otimização da geometria de moléculas com **NWCHem + ASE** no Google Colab

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares)

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

Este notebook mostra como:

1. instalar o **[NWChem](https://nwchemgit.github.io/)** e o **ASE** no Google Colab;
2. construir uma molécula de **H₂O**;
3. configurar um cálculo **DFT** com **base gaussiana** com NWChem via ASE;
4. otimizar a geometria usando o otimizador **BFGS** do ASE;
5. analisar a energia, os comprimentos O–H e o ângulo H–O–H antes e depois da otimização.


## 1. Instalação dos pacotes

O NWChem pode ser instalado no ambiente Ubuntu do Colab via `apt`.  
Também instalaremos o ASE.

In [ ]:
%%capture
!pip install ase
!apt-get install -y nwchem

In [ ]:
import os
os.environ['ASE_NWCHEM_COMMAND'] = 'nwchem PREFIX.nwi > PREFIX.nwo'

## Criando a molécula com ambiente ASE

In [ ]:
from ase import Atoms
import numpy as np

molecule_name = 'H2O'

# Definindo a molécula de interesse
my_molecule = Atoms(molecule_name, positions=[[1.5, 0, 0], [0, 1.5, 0], [0, 0, 0]])

Visualização da molécula

In [ ]:
from ase.visualize import view

view(my_molecule, viewer='x3d')

## Calculando energia da molécula

In [ ]:
from ase.calculators.nwchem import NWChem

# Configurando a calculadora NWChem
calc = NWChem(
    label='h2o',
    xc='PBE',
    basis='6-31++G**',
    set={'dft:iterations': 20}
)

# Associar a calculadora NWChem à molécula
my_molecule.calc = calc

Cálculo da energia inicial da molécula

In [ ]:
E_initial = my_molecule.get_potential_energy()

print(f"Energia inicial: {E_initial:.3f} eV")

As forças podem ser calculadas diretamente pelo teorema de Feynman-Hellman:

In [ ]:
forces_initial = my_molecule.get_forces()

print(f"Força inicial: {forces_initial}")

In [ ]:
print(f"Máxima força inicial = {np.abs(forces_initial).max():.6f} eV/Å")

## Otimização da estrutura da molécula

- Usando o método **BFGS** para otimização da geometria

- O critério de convergência será $\max_i |\mathbf{F}_i| < 0.02 \ \text{eV/Å}$


In [ ]:
from ase.optimize import BFGS

# Otimização geométrica
optimizer = BFGS(
    my_molecule,
    trajectory="h2o_optimization.traj",
    logfile="h2o_optimization.log",
)

optimizer.run(
    fmax=0.02,   # critério de convergência em eV/Å
)

Cálculo da energia após otimização

In [ ]:
E_final = my_molecule.get_potential_energy()
print(f"Energia final:   {E_final:.3f} eV")
print(f"Variação:        {E_final - E_initial:.3f} eV")

Cálculo das forças finais

In [ ]:
forces_final = my_molecule.get_forces()

print(f"Máxima força final = {np.abs(forces_final).max():.6f} eV/Å")

In [ ]:
from ase.visualize import view

view(my_molecule, viewer='x3d')

In [ ]:
my_molecule

Elementos químicos presentes na molécula

In [ ]:
my_molecule.get_chemical_symbols()

Calculando distância O-H

In [ ]:
my_molecule.get_distance(0,2)

In [ ]:
my_molecule.get_distance(1,2)

Calculando ângulo H-O-H

In [ ]:
my_molecule.get_angle(0,2,1)

Exportamos a estrutura final em formatos úteis:

- `.xyz` para visualização genérica;
- `.cif` para visualizadores cristalográficos;

In [ ]:
from ase.io import write
write(f'{molecule_name}_optimized.cif', my_molecule)
write(f'{molecule_name}_optimized.xyz', my_molecule)

## Conhecendo a Energia Potencial da Molécula (Aplicação para Campos de Forças)

Vamos agora variar a posição dos átomos e verificar como a energia potencial varia com:
- o comprimento OH
- o ângulo HOH.

In [ ]:
angleHOH = 104.87
lOH = 0.973

### Energia da ligação OH: $\ell_{OH}$


In [ ]:
lOHarray = np.linspace(0.94,1.01,11)
EOHarray = np.zeros_like(lOHarray)

for i, l in enumerate(lOHarray):
  my_molecule.set_positions([[0.,-l*np.sin(0.5*np.radians(angleHOH)), l*np.cos(0.5*np.radians(angleHOH))],
   [0.,l*np.sin(0.5*np.radians(angleHOH)), l*np.cos(0.5*np.radians(angleHOH))],
   [0,0,0]])
  EOHarray[i] = my_molecule.get_potential_energy()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(4,3))
plt.scatter(lOHarray, EOHarray)
plt.xlabel('comprimento OH (Å)')
plt.ylabel('E (eV)')
plt.show()

Fitting para campo de força

$E_\text{bond} = E_0 + n_\text{bonds}\frac{1}{2}k_b (l-l_0)^2$

In [ ]:
from scipy.optimize import curve_fit

def func(x, k, x0, E0):
    return 2*(0.5*k * (x-x0)**2) + E0 # Cuidado! Aqui temos duas ligações O-H

popt_d, pcov_d = curve_fit(func, lOHarray, EOHarray, p0=[10.0,lOHarray[EOHarray.argmin()],EOHarray.min()])

lengths_new = np.arange(lOHarray[0], lOHarray[-1]+0.001, 0.001)

In [ ]:
popt_d

In [ ]:
k_b = popt_d[0]

print(f'Constante de força k_b: {k_b:.3f} eV ou {k_b*1.6e-19*1e-3*6.022e23:.2f} kJ/mol')

In [ ]:
plt.figure(figsize=(4,3))
plt.scatter(lOHarray, EOHarray, label='data')
plt.scatter(popt_d[1],popt_d[2],color='red')
plt.plot(lengths_new, func(lengths_new, *popt_d), 'k', label='fit')
plt.legend()
plt.xlabel('comprimento OH (Å)')
plt.ylabel('E (eV)')
plt.show()

### Energia de ângulo HOH: $\theta_{HOH}$

In [ ]:
angleHOHarray = np.linspace(103,106,11)
EHOHarray = np.zeros_like(angleHOHarray)

for i, angle in enumerate(angleHOHarray):
  my_molecule.set_positions([[0.,-lOH*np.sin(0.5*np.radians(angle)), lOH*np.cos(0.5*np.radians(angle))],
   [0.,lOH*np.sin(0.5*np.radians(angle)), lOH*np.cos(0.5*np.radians(angle))],
   [0,0,0]])
  EHOHarray[i] = my_molecule.get_potential_energy()

In [ ]:
plt.figure(figsize=(4,3))
plt.scatter(angleHOHarray, EHOHarray)
plt.xlabel('ângulo HOH (º)')
plt.ylabel('E (eV)')
plt.show()

$E_\text{angle} = E_0 + \frac{1}{2}k_\theta (\theta-\theta_0)^2$

In [ ]:
def func_theta(theta, k, theta_0, E_0):
    return 0.5*k * (np.radians(theta)-np.radians(theta_0))**2 + E_0

popt_theta, pcov_theta = curve_fit(func_theta, angleHOHarray, EHOHarray, p0=[10.0,angleHOHarray[EHOHarray.argmin()],EHOHarray.min()])

angles_new = np.arange(angleHOHarray[0], angleHOHarray[-1]+0.01, 0.01)

In [ ]:
popt_theta

In [ ]:
k_theta = popt_theta[0]

print(f'Constante de força k_theta: {k_theta:.3f} eV ou {k_theta*1.6e-19*1e-3*6.022e23:.2f} kJ/mol')

In [ ]:
plt.figure(figsize=(4,3))
plt.scatter(angleHOHarray, EHOHarray, label='data')
plt.scatter(popt_theta[1],popt_theta[2],color='red')
plt.plot(angles_new, func_theta(angles_new, *popt_theta), 'k',label='fit')
plt.legend()
plt.xlabel('ângulo HOH (º)')
plt.ylabel('E (eV)')
plt.show()

## Cálculo da densidade eletrônica $n(\boldsymbol{r})$

Cria um arquivo `.cube` a ser lido por outros programas como [VESTA](https://jp-minerals.org/vesta/en/)

In [ ]:
dplot_input = f"""
restart h2o
permanent_dir ./h2o

dplot
  title "Total electron density of H2O"
  vectors ./h2o/h2o.movecs

  limitxyz
    -3.0  3.0  80
    -3.0  3.0  80
    -3.0  3.0  80

  spin total
  gaussian
  output h2o_density.cube
end

task dplot
"""

with open("h2o_density.nwi", "w") as f:
    f.write(dplot_input)

In [ ]:
!nwchem h2o_density.nwi > h2o_density.nwo

## Cálculo da Superfície de Potencial ELetrostático (ESP)

Nesse caso estamos calculando $\phi(\boldsymbol{r})$

In [ ]:
esp_input = """
restart h2o
permanent_dir ./h2o

property
  vectors ./h2o/h2o.movecs
  esp
  grid rmin -3.0 -3.0 -3.0 rmax 3.0 3.0 3.0 ngrid 81 81 81 output h2o_esp.cube
end

task dft property
"""

with open("h2o_esp.nwi", "w") as f:
    f.write(esp_input)

In [ ]:
!nwchem h2o_esp.nwi > h2o_esp.nwo

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Calcule o comprimento de ligação e o ângulo de ligação da molécula de **H2O** usando outro funcional **XC** e outro **conjunto de base**

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Calcule o comprimento de ligação e o ângulo de ligação da molécula de **CO2** usando outro funcional **XC** e outro **conjunto de base**

**<span style="color:#A03;font-size:14pt">
&#x270B; HANDS-ON! &#x1F528;
</span>**

> Escolha uma molécula **de seu interesse** e
> - otimize sua geometria
> - calcule a densidade eletrônica
> - calcule o potencial eletrostático